### Read

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

diagnosis_schema = StructType([
    StructField("diagnosis_code", StringType(), False),
    StructField("category", StringType(), True),
    StructField("severity", StringType(), True),
])

df_raw = spark.read.csv(
    "/Volumes/workspace/default/myvol/raw/diagnosis/diagnosis.csv",
    header=True,
    schema=diagnosis_schema
)
print(f"Raw rows: {df_raw.count()}")
df_raw.show(5)

### Add metadata

In [0]:
df_bronze = df_raw \
    .withColumn("_ingestion_timestamp", F.current_timestamp()) \
    .withColumn("_source_file", F.lit("diagnosis.csv")) \
    .withColumn("_source_layer", F.lit("bronze")) \
    .withColumn("_is_phi", F.lit(False))

### Write Delta

In [0]:
df_bronze.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("/Volumes/workspace/default/myvol/bronze/diagnosis/")

print("✅ Bronze diagnosis Delta table written")

### Validate

In [0]:
df_verify = spark.read.format("delta") \
    .load("/Volumes/workspace/default/myvol/bronze/diagnosis/")
assert df_verify.count() == df_raw.count(), "❌ Row count mismatch!"
print(f"✅ Validation passed — {df_verify.count()} rows")